In [ ]:
!pip install pytorch-lightning timm
import os
import random
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance, ImageOps
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from glob import glob
import cv2
from pytorch_lightning.callbacks import TQDMProgressBar
from pytorch_lightning.loggers import TensorBoardLogger

import torch
import torchvision.transforms as T
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

os.environ['KAGGLE_USERNAME'] = "birvika"
os.environ['KAGGLE_KEY'] = "15d2dbff346b377b89dab3f99d4fadd4"
import kagglehub

dataset_path = kagglehub.competition_download('dl-lab-4-ocr')
train_dir = os.path.join(dataset_path, 'train/train')
test_dir = os.path.join(dataset_path, 'test/test')
train_df = pd.read_csv(os.path.join(dataset_path, 'train.csv'))

device = "cuda" if torch.cuda.is_available() else "cpu"
seed = 9999
pl.seed_everything(seed)


from google.colab import drive
drive.mount("/content/drive")


font_path = '/content/drive/MyDrive/5ka-sans-design/5kaSansDesign-Ultra.ttf'
if not os.path.exists(font_path):
    font_path = '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf'
print("Font:", font_path)


BACKGROUNDS_FOLDER = '/content/drive/MyDrive/фоны'
backgrounds = []
if os.path.exists(BACKGROUNDS_FOLDER):
    for ext in ['*.png', '*.jpg', '*.jpeg']:
        for f in glob(os.path.join(BACKGROUNDS_FOLDER, ext)):
            img = cv2.imread(f)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                backgrounds.append(img)
    print(f"Loaded {len(backgrounds)} backgrounds")

if not backgrounds:
    for _ in range(50):
        color = tuple(random.randint(200, 255) for _ in range(3))
        backgrounds.append(np.full((128, 32, 3), color, dtype=np.uint8))


def clean_text(text):

    text = ''.join([c for c in str(text) if c.isdigit()])
    return text if text != "" else "0"

def add_white_stripes(img, p=0.12):

    if random.random() > p:
        return img

    img_array = np.array(img).astype(np.float32)
    h, w = img_array.shape[:2]


    if random.choice(['stripe', 'stripe', 'spot']) == 'stripe':

        if random.choice(['horizontal', 'vertical']) == 'horizontal':
            y = random.randint(0, h - 1)
            thickness = random.randint(2, 5)
            y_start = max(0, y - thickness // 2)
            y_end = min(h, y + thickness // 2)
            img_array[y_start:y_end, :] = 255
        else:
            x = random.randint(0, w - 1)
            thickness = random.randint(2, 5)
            x_start = max(0, x - thickness // 2)
            x_end = min(w, x + thickness // 2)
            img_array[:, x_start:x_end] = 255
    else:

        cx = random.randint(0, w - 1)
        cy = random.randint(0, h - 1)
        radius = random.randint(5, 12)
        Y, X = np.ogrid[:h, :w]
        mask = (X - cx) ** 2 + (Y - cy) ** 2 <= radius ** 2
        img_array[mask] = np.clip(img_array[mask] * 1.5, 0, 255)

    return Image.fromarray(img_array.astype(np.uint8))

def generate_synthetic():

    img_w, img_h = 128, 64
    bg = random.choice(backgrounds).copy()
    bg = cv2.resize(bg, (img_w, img_h))
    img = Image.fromarray(bg)
    draw = ImageDraw.Draw(img)


    num_digits = random.choices([1, 2, 3, 4, 5], weights=[5, 10, 20, 50, 10], k=1)[0]

    if num_digits == 1:
        price = str(random.randint(1, 9))
        font_size = random.randint(38, 48)
    elif num_digits == 2:
        price = str(random.randint(10, 99))
        font_size = random.randint(34, 44)
    elif num_digits == 3:
        price = str(random.randint(100, 999))
        font_size = random.randint(30, 40)
    elif num_digits == 4:
        price = str(random.randint(1000, 9999))
        font_size = random.randint(26, 36)
    elif num_digits == 5:
        price = str(random.randint(10000, 99999))
        font_size = random.randint(22, 30)

    try:
        font = ImageFont.truetype(font_path, font_size)
    except:
        font = ImageFont.load_default()

    bbox = draw.textbbox((0, 0), price, font=font)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]
    x = (img_w - text_w) // 2 + random.randint(-5, 5)
    y = (img_h - text_h) // 2 + random.randint(-5, 5)
    draw.text((x, y), price, fill=(0, 0, 0), font=font)

    return img, price, (0, 0, 0)

def degrade(img, bg_color):

    if random.random() < 0.5:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.5)))

    if random.random() < 0.5:
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.7, 1.3))

    if random.random() < 0.5:
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.7, 1.3))
    return img


from torch.utils.data import DataLoader

parseq_transform = T.Compose([
    T.Resize((32, 128), T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

class PARSeqDataset(torch.utils.data.Dataset):
    def __init__(self, df, root_dir, num_synthetic=6000):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.num_synthetic = num_synthetic

    def __len__(self):
        return len(self.df) + self.num_synthetic

    def __getitem__(self, idx):

        if idx < len(self.df):
            filename = self.df.iloc[idx]['Filename']
            price = clean_text(self.df.iloc[idx]['Price'])
            image = Image.open(os.path.join(self.root_dir, filename)).convert("RGB")
            image = ImageOps.autocontrast(image)
            if random.random() < 0.3:
                image = ImageEnhance.Brightness(image).enhance(random.uniform(0.8, 1.2))

        else:
            image, price, bg_color = generate_synthetic()
            image = degrade(image, bg_color)

            image = add_white_stripes(image, p=0.12)

            if random.random() < 0.8:
                w_sm, h_sm = random.randint(20, 40), random.randint(10, 20)
                image = image.resize((w_sm, h_sm), Image.BILINEAR)

        image = image.resize((128, 64), Image.BICUBIC)
        image_tensor = parseq_transform(image)
        return {"image": image_tensor, "label": price}

def collate_fn(batch):
    images = torch.stack([item["image"] for item in batch])
    labels = [item["label"] for item in batch]
    return images, labels


train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42)

train_dataset = PARSeqDataset(train_df, train_dir, num_synthetic=7000)
val_dataset = PARSeqDataset(val_df, train_dir, num_synthetic=0)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn, num_workers=2)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 17.0 MB/s eta 0:00:00


100%|██████████| 19.6M/19.6M [00:00<00:00, 38.4MB/s]

Extracting files...



INFO:lightning_fabric.utilities.seed:Seed set to 9999


Mounted at /content/drive
Font: /content/drive/MyDrive/5ka-sans-design/5kaSansDesign-Ultra.ttf
Loaded 0 backgrounds
Train batches: 298, Val batches: 48
Train samples: 19040, Val samples: 3010


In [ ]:

sample = train_dataset[0]
print(f"Тип: {type(sample)}")
print(f"Ключи: {sample.keys() if isinstance(sample, dict) else 'Not a dict'}")
print(f"Значение: {sample}")

idx = len(train_df) + 1
sample_synth = train_dataset[idx]
print(f"\nСинтетический пример: {sample_synth}")

Тип: <class 'dict'>
Ключи: dict_keys(['image', 'label'])
Значение: {'image': tensor([[[0.6000, 0.5843, 0.5765,  ..., 0.4588, 0.4510, 0.4431],
         [0.5686, 0.5608, 0.5529,  ..., 0.4824, 0.4745, 0.4667],
         [0.5216, 0.5216, 0.5137,  ..., 0.5059, 0.4980, 0.4902],
         ...,
         [0.5137, 0.5059, 0.4980,  ..., 0.4667, 0.4745, 0.4745],
         [0.4588, 0.4667, 0.4588,  ..., 0.4667, 0.4745, 0.4745],
         [0.4118, 0.4118, 0.4039,  ..., 0.4431, 0.4510, 0.4510]],

        [[0.7255, 0.7098, 0.7020,  ..., 0.5922, 0.5843, 0.5843],
         [0.6941, 0.6863, 0.6784,  ..., 0.6235, 0.6157, 0.6078],
         [0.6471, 0.6471, 0.6392,  ..., 0.6471, 0.6392, 0.6392],
         ...,
         [0.6863, 0.6863, 0.6784,  ..., 0.5451, 0.5608, 0.5686],
         [0.6549, 0.6549, 0.6471,  ..., 0.5451, 0.5608, 0.5686],
         [0.6157, 0.6157, 0.6078,  ..., 0.5216, 0.5373, 0.5451]],

        [[0.7255, 0.7176, 0.6941,  ..., 0.6549, 0.6392, 0.6392],
         [0.6863, 0.6863, 0.6706,  ..., 0.6863

In [ ]:
import torch
import pytorch_lightning as pl
from torch.optim.lr_scheduler import ReduceLROnPlateau
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, Callback
from pytorch_lightning.callbacks.progress import TQDMProgressBar
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import os
import torch.nn.functional as F
from pytorch_lightning.loggers import TensorBoardLogger


SEED = 42
pl.seed_everything(SEED)

train_df_seed, val_df_seed = train_test_split(train_df, test_size=0.2, random_state=SEED)


train_dataset = PARSeqDataset(train_df_seed, train_dir, num_synthetic=7000)
val_dataset = PARSeqDataset(val_df_seed, train_dir, num_synthetic=0)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn, num_workers=2)


model = torch.hub.load('baudm/parseq', 'parseq', pretrained=True)
model = model.to(device)
model.hparams.lr = 4e-5


save_dir = '/content/drive/MyDrive/laba_go_4/seed_42/'
os.makedirs(save_dir, exist_ok=True)
print(f"Папка создана: {save_dir}")

if os.access(save_dir, os.W_OK):
    print("Есть права на запись в папку")
else:
    print("Нет прав на запись в папку!")

checkpoint_callback = ModelCheckpoint(
    dirpath=save_dir,
    filename='parseq_best_seed42',
    monitor='val_accuracy',
    mode='max',
    save_top_k=1,
)

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=4,
    mode='max',
    verbose=True,
    min_delta=0.001,
)

class LRSchedulerCallback(Callback):
    def __init__(self, patience=2, factor=0.5):
        self.patience = patience
        self.factor = factor
        self.best_acc = 0
        self.counter = 0

    def on_validation_epoch_end(self, trainer, pl_module):
        current_acc = trainer.callback_metrics.get('val_accuracy', 0)
        if isinstance(current_acc, torch.Tensor):
            current_acc = current_acc.item()

        if current_acc > self.best_acc:
            self.best_acc = current_acc
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                old_lr = trainer.optimizers[0].param_groups[0]['lr']
                new_lr = old_lr * self.factor
                for param_group in trainer.optimizers[0].param_groups:
                    param_group['lr'] = new_lr
                print(f"\nУменьшен LR: {old_lr:.2e} -> {new_lr:.2e}")
                self.counter = 0

logger = TensorBoardLogger("lightning_logs", name="parseq_seed42")

if torch.cuda.is_available():
    trainer = pl.Trainer(
        max_epochs=30,
        accelerator='gpu',
        devices=1,
        callbacks=[
            checkpoint_callback,
            early_stop,
            LRSchedulerCallback(patience=2, factor=0.5),
            TQDMProgressBar(refresh_rate=10)
        ],
        enable_model_summary=False
    )
else:
    trainer = pl.Trainer(
        max_epochs=30,
        accelerator='cpu',
        callbacks=[
            checkpoint_callback,
            early_stop,
            LRSchedulerCallback(patience=2, factor=0.5),
            TQDMProgressBar(refresh_rate=10)
        ],
        enable_model_summary=False
    )

print(f"Learning rate: {model.hparams.lr}")
print("Early Stopping: остановка если 4 эпохи без улучшения")
print("LR Scheduler: уменьшение LR в 2 раза если 2 эпохи без улучшений")
print(f"Модель сохраняется в: {save_dir}")

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

print("\n" + "="*50)
print("РЕЗУЛЬТАТЫ ОБУЧЕНИЯ SEED=42")
print("="*50)

if early_stop.stopped_epoch > 0:
    print(f"Early Stopping сработал на эпохе {early_stop.stopped_epoch}")
    print(f"Лучшая val_accuracy: {early_stop.best_score:.4f} ({early_stop.best_score*100:.2f}%)")
else:
    print(f"Обучение завершило все {trainer.current_epoch} эпох")
    print(f"Лучшая val_accuracy: {early_stop.best_score:.4f} ({early_stop.best_score*100:.2f}%)")

best_model_path = checkpoint_callback.best_model_path
print(f"Лучшая модель сохранена: {best_model_path}")

INFO:lightning_fabric.utilities.seed:Seed set to 42
Using cache found in /root/.cache/torch/hub/baudm_parseq_main
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.utilities.rank_zero:Loading `train_dataloader` to estimate number of stepping batches.


Папка создана: /content/drive/MyDrive/laba_go_4/seed_42/
Есть права на запись в папку
Learning rate: 4e-05
Early Stopping: остановка если 4 эпохи без улучшения
LR Scheduler: уменьшение LR в 2 раза если 2 эпохи без улучшений
Модель сохраняется в: /content/drive/MyDrive/laba_go_4/seed_42/


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]


Уменьшен LR: 1.04e-03 -> 5.20e-04


Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.callbacks.early_stopping:Monitored metric val_accuracy did not improve in the last 4 records. Best score: 99.751. Signaling Trainer to stop.



РЕЗУЛЬТАТЫ ОБУЧЕНИЯ SEED=42
Early Stopping сработал на эпохе 4
Лучшая val_accuracy: 99.7508 (9975.08%)
Лучшая модель сохранена: /content/drive/MyDrive/laba_go_4/seed_42/parseq_best_seed42.ckpt


In [ ]:
import torch
import pytorch_lightning as pl
from torch.optim.lr_scheduler import ReduceLROnPlateau
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, Callback
from pytorch_lightning.callbacks.progress import TQDMProgressBar
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import os
import torch.nn.functional as F
from pytorch_lightning.loggers import TensorBoardLogger
import gc


torch.cuda.empty_cache()
gc.collect()


SEED = 9999
pl.seed_everything(SEED)


train_df_seed, val_df_seed = train_test_split(train_df, test_size=0.2, random_state=SEED)


train_dataset = PARSeqDataset(train_df_seed, train_dir, num_synthetic=7000)
val_dataset = PARSeqDataset(val_df_seed, train_dir, num_synthetic=0)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn, num_workers=2)


model = torch.hub.load('baudm/parseq', 'parseq', pretrained=True)
model = model.to(device)
model.hparams.lr = 4e-5


save_dir = '/content/drive/MyDrive/laba_go_4/seed_9999/'
os.makedirs(save_dir, exist_ok=True)
print(f"Папка создана: {save_dir}")

if os.access(save_dir, os.W_OK):
    print("Есть права на запись в папку")
else:
    print("Нет прав на запись в папку!")

checkpoint_callback = ModelCheckpoint(
    dirpath=save_dir,
    filename='parseq_best_seed9999',
    monitor='val_accuracy',
    mode='max',
    save_top_k=1,
)

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=4,
    mode='max',
    verbose=True,
    min_delta=0.001,
)

class LRSchedulerCallback(Callback):
    def __init__(self, patience=2, factor=0.5):
        self.patience = patience
        self.factor = factor
        self.best_acc = 0
        self.counter = 0

    def on_validation_epoch_end(self, trainer, pl_module):
        current_acc = trainer.callback_metrics.get('val_accuracy', 0)
        if isinstance(current_acc, torch.Tensor):
            current_acc = current_acc.item()

        if current_acc > self.best_acc:
            self.best_acc = current_acc
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                old_lr = trainer.optimizers[0].param_groups[0]['lr']
                new_lr = old_lr * self.factor
                for param_group in trainer.optimizers[0].param_groups:
                    param_group['lr'] = new_lr
                print(f"\nУменьшен LR: {old_lr:.2e} -> {new_lr:.2e}")
                self.counter = 0

logger = TensorBoardLogger("lightning_logs", name="parseq_seed9999")

if torch.cuda.is_available():
    trainer = pl.Trainer(
        max_epochs=30,
        accelerator='gpu',
        devices=1,
        callbacks=[
            checkpoint_callback,
            early_stop,
            LRSchedulerCallback(patience=2, factor=0.5),
            TQDMProgressBar(refresh_rate=10)
        ],
        enable_model_summary=False
    )
else:
    trainer = pl.Trainer(
        max_epochs=30,
        accelerator='cpu',
        callbacks=[
            checkpoint_callback,
            early_stop,
            LRSchedulerCallback(patience=2, factor=0.5),
            TQDMProgressBar(refresh_rate=10)
        ],
        enable_model_summary=False
    )

print(f"Learning rate: {model.hparams.lr}")
print("Early Stopping: остановка если 4 эпохи без улучшения")
print("LR Scheduler: уменьшение LR в 2 раза если 2 эпохи без улучшений")
print(f"Модель сохраняется в: {save_dir}")

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

print("\n" + "="*50)
print("РЕЗУЛЬТАТЫ ОБУЧЕНИЯ SEED=9999")
print("="*50)

if early_stop.stopped_epoch > 0:
    print(f"Early Stopping сработал на эпохе {early_stop.stopped_epoch}")
    print(f"Лучшая val_accuracy: {early_stop.best_score:.4f} ({early_stop.best_score*100:.2f}%)")
else:
    print(f"Обучение завершило все {trainer.current_epoch} эпох")
    print(f"Лучшая val_accuracy: {early_stop.best_score:.4f} ({early_stop.best_score*100:.2f}%)")

best_model_path = checkpoint_callback.best_model_path
print(f"Лучшая модель сохранена: {best_model_path}")

INFO:lightning_fabric.utilities.seed:Seed set to 9999
Using cache found in /root/.cache/torch/hub/baudm_parseq_main
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.utilities.rank_zero:Loading `train_dataloader` to estimate number of stepping batches.


Папка создана: /content/drive/MyDrive/laba_go_4/seed_9999/
Есть права на запись в папку
Learning rate: 4e-05
Early Stopping: остановка если 4 эпохи без улучшения
LR Scheduler: уменьшение LR в 2 раза если 2 эпохи без улучшений
Модель сохраняется в: /content/drive/MyDrive/laba_go_4/seed_9999/


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]


Уменьшен LR: 9.43e-04 -> 4.71e-04


Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.callbacks.early_stopping:Monitored metric val_accuracy did not improve in the last 4 records. Best score: 99.626. Signaling Trainer to stop.



РЕЗУЛЬТАТЫ ОБУЧЕНИЯ SEED=9999
Early Stopping сработал на эпохе 8
Лучшая val_accuracy: 99.6262 (9962.62%)
Лучшая модель сохранена: /content/drive/MyDrive/laba_go_4/seed_9999/parseq_best_seed9999.ckpt


In [ ]:
import glob
import pandas as pd
from PIL import Image, ImageFilter
from tqdm import tqdm
from collections import Counter

print("ЗАГРУЗКА МОДЕЛЕЙ ДЛЯ АНСАМБЛЯ")

ckpt_42 = glob.glob('/content/drive/MyDrive/laba_go_4/seed_42/*.ckpt')[0]
ckpt_9999 = glob.glob('/content/drive/MyDrive/laba_go_4/seed_9999/*.ckpt')[0]

print(f"Загружаем seed_42: {ckpt_42}")
print(f"Загружаем seed_9999: {ckpt_9999}")

model_42 = torch.hub.load('baudm/parseq', 'parseq', pretrained=True).to(device)
model_9999 = torch.hub.load('baudm/parseq', 'parseq', pretrained=True).to(device)

model_42.load_state_dict(torch.load(ckpt_42, map_location=device)['state_dict'])
model_9999.load_state_dict(torch.load(ckpt_9999, map_location=device)['state_dict'])

model_42.eval()
model_9999.eval()
print("Обе модели загружены")

def predict_ensemble_tta(image, model1, model2, device):
    variants = [image, image.filter(ImageFilter.GaussianBlur(0.4))]
    batch = torch.stack([parseq_transform(img) for img in variants]).to(device)
    with torch.no_grad():
        logits1 = model1(batch)
        logits2 = model2(batch)
        preds1 = logits1.softmax(-1)
        preds2 = logits2.softmax(-1)
        texts1, _ = model1.tokenizer.decode(preds1)
        texts2, _ = model2.tokenizer.decode(preds2)
    texts1 = [clean_text(t) for t in texts1]
    texts2 = [clean_text(t) for t in texts2]
    all_texts = texts1 + texts2
    if all(t == "" for t in all_texts):
        return "0"
    return Counter(all_texts).most_common(1)[0][0]

print("\nПРЕДСКАЗАНИЯ АНСАМБЛЯ (HARD VOTING)")

if 'dataset_path' not in dir():
    import kagglehub
    dataset_path = kagglehub.competition_download('dl-lab-4-ocr')
test_dir = os.path.join(dataset_path, 'test/test')
submission = pd.read_csv(os.path.join(dataset_path, 'sample_submission.csv'))

pred_labels_ensemble = []
for filename in tqdm(submission["Filename"], desc="Ансамбль"):
    image_path = os.path.join(test_dir, filename)
    image = Image.open(image_path).convert("RGB")
    pred = predict_ensemble_tta(image, model_42, model_9999, device)
    pred_labels_ensemble.append(pred)

submission["Price"] = pred_labels_ensemble
submission_save_path = '/content/drive/MyDrive/laba_go_4/submission_ensemble_hard_voting.csv'
submission.to_csv(submission_save_path, index=False)
print(f"\nАнсамбль сохранен: {submission_save_path}")

pred_counter = Counter(pred_labels_ensemble)
print(f"\nСтатистика предсказаний ансамбля:")
print(f"  Уникальных значений: {len(pred_counter)}")
print(f"  Наиболее частые:")
for value, count in pred_counter.most_common(5):
    print(f"    {value}: {count} ({count/len(pred_labels_ensemble)*100:.1f}%)")

ЗАГРУЗКА МОДЕЛЕЙ ДЛЯ АНСАМБЛЯ
Загружаем seed_42: /content/drive/MyDrive/laba_go_4/seed_42/parseq_best_seed42.ckpt
Загружаем seed_9999: /content/drive/MyDrive/laba_go_4/seed_9999/parseq_best_seed9999.ckpt


Using cache found in /root/.cache/torch/hub/baudm_parseq_main
Using cache found in /root/.cache/torch/hub/baudm_parseq_main


Обе модели загружены

ПРЕДСКАЗАНИЯ АНСАМБЛЯ (HARD VOTING)



Ансамбль: 100%|██████████| 3762/3762 [02:36<00:00, 24.01it/s]



Ансамбль сохранен: /content/drive/MyDrive/laba_go_4/submission_ensemble_hard_voting.csv

Статистика предсказаний ансамбля:
  Уникальных значений: 360
  Наиболее частые:
    109: 291 (7.7%)
    199: 192 (5.1%)
    189: 166 (4.4%)
    139: 160 (4.3%)
    32: 115 (3.1%)


In [1]:
# import os
# import numpy as np
# from PIL import Image
# from tqdm import tqdm



# save_dir = '/content/drive/MyDrive/laba_go_4/'
# synth_images_dir = os.path.join(save_dir, 'synthetic_images')
# os.makedirs(synth_images_dir, exist_ok=True)

# print(f"📁 Папка для синтетики: {synth_images_dir}")
# print(f"📊 Всего синтетических изображений: {train_dataset.num_synthetic}")
# print(f"🎨 Начинаем сохранение...")


# def save_all_synthetic(dataset, output_dir):


#     real_count = len(dataset.df) if hasattr(dataset, 'df') else 0


#     print(f"Реальных изображений: {real_count}")
#     print(f"Синтетических: {dataset.num_synthetic}")

#     saved_count = 0
#     failed_count = 0

#     for idx in tqdm(range(real_count, real_count + dataset.num_synthetic), desc="Сохранение синтетики"):
#         try:
#             item = dataset[idx]
#             image_tensor = item['image']
#             label = item['label']


#             img = image_tensor.permute(1, 2, 0).cpu().numpy()
#             img = img * 0.5 + 0.5
#             img = np.clip(img, 0, 1)
#             img = (img * 255).astype(np.uint8)


#             filename = f"synthetic_{saved_count:05d}_price_{label}.png"
#             img_pil = Image.fromarray(img)
#             img_pil.save(os.path.join(output_dir, filename))
#             saved_count += 1

#         except Exception as e:
#             failed_count += 1
#             print(f"Ошибка при сохранении индекса {idx}: {e}")

#     print(f"\n✅ Сохранено {saved_count} синтетических изображений")
#     print(f"❌ Ошибок: {failed_count}")
#     print(f"📁 Путь: {output_dir}")


# save_all_synthetic(train_dataset, synth_images_dir)


# preview_dir = os.path.join(save_dir, 'synthetic_preview')
# os.makedirs(preview_dir, exist_ok=True)

# def save_preview_images(dataset, num_samples=100, output_dir=preview_dir):
#     """Сохраняет несколько синтетических примеров для предпросмотра"""

#     real_count = len(dataset.df) if hasattr(dataset, 'df') else 0

#     for i in tqdm(range(min(num_samples, dataset.num_synthetic)), desc="Сохранение превью"):
#         idx = real_count + i
#         item = dataset[idx]

#         img = item['image'].permute(1, 2, 0).cpu().numpy()
#         img = img * 0.5 + 0.5
#         img = np.clip(img, 0, 1)
#         img = (img * 255).astype(np.uint8)

#         img_pil = Image.fromarray(img)
#         filename = f"preview_{i:04d}_price_{item['label']}.png"
#         img_pil.save(os.path.join(output_dir, filename))

#     print(f"✅ Сохранено {num_samples} примеров в {output_dir}")


# save_preview_images(train_dataset, num_samples=100)



# print("\n" + "="*50)
# print("СТАТИСТИКА СОХРАНЕНИЯ")
# print("="*50)


# import glob
# synth_files = glob.glob(os.path.join(synth_images_dir, "*.png"))
# preview_files = glob.glob(os.path.join(preview_dir, "*.png"))

# print(f"📁 {synth_images_dir}: {len(synth_files)} файлов")
# print(f"📁 {preview_dir}: {len(preview_files)} файлов")
# print(f"💾 Общий размер: ~{len(synth_files) * 50 / 1024:.1f} MB")


# if synth_files:
#     print(f"\n📋 Примеры имен файлов:")
#     for f in synth_files[:5]:
#         print(f"   {os.path.basename(f)}")